# Interpretando relações: normalização, confundimento e agregação

Na etapa anterior, aprendemos a **identificar, visualizar e quantificar associações** entre variáveis. Agora a pergunta muda:

> **uma associação observada significa exatamente aquilo que parece significar?**

Vamos reutilizar o corpus `dados/documentos_relacoes.csv`, gerado na etapa **01a**. O arquivo original
`dados/documentos.csv` continua intocado.

Nesta etapa, investigaremos quatro fontes frequentes de interpretações enganosas:

- **contagens e exposição**: documentos maiores têm mais oportunidades de conter ocorrências;
- **terceiras variáveis**: duas medidas podem variar juntas porque ambas dependem de outra característica;
- **agregação**: uma tendência no corpus inteiro pode ser diferente das tendências dentro de grupos;
- **dados ausentes**: não basta saber quantos valores faltam; importa saber **onde** eles faltam.

O objetivo não é aprender novas técnicas sofisticadas, mas desenvolver um hábito: **antes de explicar uma relação,
tentar descobrir o que poderia alterar sua interpretação**.


In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import os
import subprocess

URL_REPOSITORIO = "https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais.git"
REPOSITORIO = Path("/content/disciplina_computacao_aplicada_humanidades_digitais")
PASTA_UNIDADE = REPOSITORIO / "unidade_04"

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             URL_REPOSITORIO, str(REPOSITORIO)],
            check=True,
        )
    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")


## Carregando o corpus

Esta etapa **não gera novamente** os dados. Isso é deliberado: a etapa 01a é responsável pela criação do corpus,
e as etapas seguintes trabalham sobre o mesmo arquivo.

Se o arquivo não existir, execute primeiro `01a_relacoes_entre_variaveis.ipynb`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ARQUIVO = Path("dados/documentos_relacoes.csv")

if not ARQUIVO.exists():
    raise FileNotFoundError(
        "O arquivo dados/documentos_relacoes.csv não foi encontrado. "
        "Execute primeiro a etapa 01a — Relações entre variáveis."
    )

dados = pd.read_csv(ARQUIVO)
print("Dimensões:", dados.shape)
dados.head()


## 1. Uma relação convincente

Comecemos por uma afirmação que parece bastante natural:

> **Documentos com mais páginas tendem a mencionar mais pessoas.**

Na etapa anterior, vimos que `paginas` e `pessoas` apresentam associação positiva. Vamos recuperar o gráfico e o
coeficiente.


In [ ]:
r_paginas_pessoas = dados["paginas"].corr(dados["pessoas"])

plt.figure(figsize=(8, 5))
plt.scatter(dados["paginas"], dados["pessoas"], alpha=0.7)
plt.xlabel("Número de páginas")
plt.ylabel("Pessoas mencionadas")
plt.title(f"Páginas × pessoas (r = {r_paginas_pessoas:.3f})")
plt.grid(alpha=0.2)
plt.show()

print(f"Correlação entre páginas e pessoas: {r_paginas_pessoas:.3f}")


O resultado descreve corretamente os dados: documentos com mais páginas **tendem** a apresentar mais menções a
pessoas.

Mas há uma pergunta anterior a qualquer explicação:

> **ter mais páginas é a característica relevante, ou as duas medidas estão respondendo ao tamanho do documento?**

Uma página adicional não precisa, por si só, produzir novas menções. Um documento maior simplesmente oferece
**mais texto — e, portanto, mais oportunidades — para que pessoas sejam mencionadas**.


## 2. Procurando uma terceira variável

Compare agora três relações:

- `palavras × paginas`;
- `palavras × pessoas`;
- `paginas × pessoas`.

Se `palavras` representa a extensão textual do documento, ela pode ajudar a entender por que as outras duas
variáveis se movem juntas.


In [ ]:
pares = [
    ("palavras", "paginas"),
    ("palavras", "pessoas"),
    ("paginas", "pessoas"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (x, y) in zip(axes, pares):
    r = dados[x].corr(dados[y])
    ax.scatter(dados[x], dados[y], alpha=0.7)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{x} × {y}\nr = {r:.3f}")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


Uma maneira útil de pensar sobre o problema é:

\[
\text{páginas} \leftarrow \text{tamanho do documento} \rightarrow \text{pessoas mencionadas}
\]

Esse esquema **não prova** que `palavras` seja a única explicação, nem é um modelo causal completo. Ele serve para
formular uma hipótese crítica: talvez parte importante da associação `paginas × pessoas` seja consequência de
ambas as variáveis crescerem com a extensão do documento.

Em estudos de corpus, isso é muito comum. Contagens de palavras, entidades, citações, nomes, tópicos ou ocorrências
dependem também de **quanto texto estava disponível para que essas ocorrências aparecessem**.


## 3. Contagens e exposição: normalizando por tamanho

Uma contagem absoluta responde a:

> **quantas pessoas foram mencionadas?**

Mas podemos formular outra pergunta:

> **com que frequência pessoas foram mencionadas, considerando a quantidade de texto disponível?**

Vamos calcular o número de pessoas mencionadas por **1.000 palavras**:

\[
\text{pessoas por 1.000 palavras}
=
1000 \times
\frac{\text{pessoas}}{\text{palavras}}.
\]

Essa transformação é uma **normalização por exposição**. O denominador representa, aproximadamente, a oportunidade
de ocorrência.


In [ ]:
dados["pessoas_por_1000_palavras"] = (
    1000 * dados["pessoas"] / dados["palavras"]
)

dados[[
    "id_documento", "palavras", "paginas",
    "pessoas", "pessoas_por_1000_palavras"
]].head()


Compare agora duas perguntas diferentes:

1. documentos com mais páginas mencionam **mais pessoas em termos absolutos**?
2. documentos com mais páginas apresentam **maior frequência de pessoas por quantidade de texto**?


In [ ]:
r_contagem = dados["paginas"].corr(dados["pessoas"])
r_taxa = dados["paginas"].corr(dados["pessoas_por_1000_palavras"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(dados["paginas"], dados["pessoas"], alpha=0.7)
axes[0].set_xlabel("Número de páginas")
axes[0].set_ylabel("Pessoas mencionadas")
axes[0].set_title(f"Contagem absoluta (r = {r_contagem:.3f})")
axes[0].grid(alpha=0.2)

axes[1].scatter(dados["paginas"], dados["pessoas_por_1000_palavras"], alpha=0.7)
axes[1].set_xlabel("Número de páginas")
axes[1].set_ylabel("Pessoas por 1.000 palavras")
axes[1].set_title(f"Contagem normalizada (r = {r_taxa:.3f})")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Páginas × pessoas: {r_contagem:.3f}")
print(f"Páginas × pessoas por 1.000 palavras: {r_taxa:.3f}")


A diferença é substantiva. No corpus, a associação entre páginas e a **contagem** de pessoas é positiva, enquanto
a associação entre páginas e a **frequência por 1.000 palavras** fica próxima de zero.

Isso não torna a contagem absoluta “errada”. As duas medidas respondem a perguntas diferentes:

- **contagem**: quantas ocorrências existem no documento?
- **taxa normalizada**: quantas ocorrências existem em relação à oportunidade de ocorrência?

Em Humanidades Digitais, escolher entre contagem e taxa é uma decisão de interpretação, não apenas uma operação
matemática.


## 4. A normalização também pode alterar comparações entre grupos

Compare os gêneros documentais usando primeiro a contagem de pessoas e depois a frequência por 1.000 palavras.
O objetivo não é procurar uma “medida vencedora”, mas observar que **mudar a pergunta pode mudar a comparação**.


In [ ]:
comparacao_generos = (
    dados.groupby("genero")
    .agg(
        n=("id_documento", "count"),
        palavras_medias=("palavras", "mean"),
        pessoas_media=("pessoas", "mean"),
        pessoas_por_1000_media=("pessoas_por_1000_palavras", "mean"),
    )
    .round(2)
)

comparacao_generos


Ao interpretar a tabela, pergunte:

- um gênero apresenta mais pessoas porque seus documentos são maiores?
- a diferença permanece quando consideramos a quantidade de texto?
- contagem e frequência estão respondendo à mesma pergunta?

O princípio é geral: antes de comparar grupos por uma contagem, procure um possível **denominador de exposição**.


## 5. Quando o corpus inteiro conta uma história diferente dos grupos

Voltemos agora à relação entre `ano` e `palavras`.

Se analisarmos todos os documentos juntos, qual tendência aparece?


In [ ]:
r_global = dados["ano"].corr(dados["palavras"])

coef_global = np.polyfit(dados["ano"], dados["palavras"], 1)
x_global = np.linspace(dados["ano"].min(), dados["ano"].max(), 100)

plt.figure(figsize=(8, 5))
plt.scatter(dados["ano"], dados["palavras"], alpha=0.65)
plt.plot(x_global, np.polyval(coef_global, x_global))
plt.xlabel("Ano")
plt.ylabel("Número de palavras")
plt.title(f"Ano × palavras — corpus inteiro (r = {r_global:.3f})")
plt.grid(alpha=0.2)
plt.show()

print(f"Correlação global: {r_global:.3f}")


O resultado agregado sugere uma tendência positiva: nos anos mais recentes do corpus, os documentos parecem ser,
em média, maiores.

Agora separe os documentos por `local`.


In [ ]:
correlacoes_por_local = (
    dados.groupby("local")
    .apply(lambda grupo: grupo["ano"].corr(grupo["palavras"]), include_groups=False)
    .rename("r_ano_palavras")
    .round(3)
)

correlacoes_por_local


In [ ]:
plt.figure(figsize=(9, 6))

for local, grupo in dados.groupby("local"):
    plt.scatter(grupo["ano"], grupo["palavras"], alpha=0.65, label=local)

    coef = np.polyfit(grupo["ano"], grupo["palavras"], 1)
    x = np.linspace(grupo["ano"].min(), grupo["ano"].max(), 100)
    plt.plot(x, np.polyval(coef, x), label=f"Tendência — {local}")

# tendência agregada
plt.plot(
    x_global,
    np.polyval(coef_global, x_global),
    linestyle="--",
    linewidth=2,
    label="Tendência — corpus inteiro",
)

plt.xlabel("Ano")
plt.ylabel("Número de palavras")
plt.title("A tendência agregada e as tendências dentro de cada local")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print(f"Corpus inteiro: {r_global:.3f}")
for local, r in correlacoes_por_local.items():
    print(f"{local}: {r:.3f}")


O sinal mudou.

- no **corpus inteiro**, `ano × palavras` apresenta associação positiva;
- entre documentos da **Capital**, a associação é negativa;
- entre documentos do **Interior**, a associação também é negativa.

Não há contradição matemática. Estamos comparando distribuições diferentes: uma relação calculada depois de
misturar grupos não precisa coincidir com as relações observadas dentro desses grupos.


## 6. Paradoxo de Simpson

**Paradoxo de Simpson** ocorre quando uma associação observada em dados agregados **desaparece ou muda de direção**
quando os dados são separados em grupos relevantes.

Neste corpus:

\[
\text{todos os documentos: ano} \uparrow \;\Rightarrow\; \text{palavras} \uparrow
\]

mas:

\[
\text{Capital: ano} \uparrow \;\Rightarrow\; \text{palavras} \downarrow
\]

e:

\[
\text{Interior: ano} \uparrow \;\Rightarrow\; \text{palavras} \downarrow.
\]

A explicação está na **composição do corpus**. Documentos da Capital tendem a ser maiores e sua participação no
corpus aumenta ao longo do período. Ao misturar os locais, essa mudança de composição é suficiente para produzir
uma tendência agregada positiva, mesmo que dentro de cada local a tendência seja negativa.

Portanto, o paradoxo não é uma falha da correlação. É um alerta de que **agregar grupos pode alterar a pergunta que
está sendo respondida**.

**Referência clássica**

Simpson, E. H. (1951). *The Interpretation of Interaction in Contingency Tables*.  
*Journal of the Royal Statistical Society: Series B (Methodological)*, 13(2), 238–241.  
DOI: 10.1111/j.2517-6161.1951.tb00088.x.


### O que mudou na composição?

Para tornar o mecanismo visível, observe a proporção de documentos da Capital em cada ano.


In [ ]:
composicao = pd.crosstab(dados["ano"], dados["local"])
composicao["proporcao_Capital"] = (
    composicao["Capital"] / composicao[["Capital", "Interior"]].sum(axis=1)
)

display(composicao)

plt.figure(figsize=(9, 4.5))
plt.plot(
    composicao.index,
    composicao["proporcao_Capital"],
    marker="o",
)
plt.ylim(0, 1)
plt.xlabel("Ano")
plt.ylabel("Proporção de documentos da Capital")
plt.title("Mudança da composição do corpus ao longo do tempo")
plt.grid(alpha=0.2)
plt.show()


A lição não é “sempre separar os dados em todos os grupos possíveis”. Separar indiscriminadamente também pode
produzir análises frágeis. A pergunta adequada é:

> **há uma variável substantivamente relevante cuja distribuição muda e que pode alterar a interpretação da
> relação agregada?**

Aqui, `local` satisfaz esse critério porque está relacionado tanto ao período quanto ao tamanho dos documentos.


## 7. Dados ausentes: quantos faltam e onde faltam?

Até agora o corpus está completo. Para estudar dados ausentes sem contaminar o arquivo original, criaremos uma
**cópia** e simularemos perdas de informação.

Imagine que parte das contagens de `pessoas` não pudesse ser recuperada em determinados documentos. Saber apenas
a porcentagem total de valores ausentes seria suficiente?


In [ ]:
dados_incompletos = dados.copy()

# Experimento reprodutível de ausência:
# concentraremos perdas em documentos antigos do Interior e acrescentaremos
# poucas perdas dispersas no restante do corpus.
rng_missing = np.random.default_rng(20261059)

candidatos_concentrados = dados_incompletos.index[
    (dados_incompletos["ano"] <= 1899)
    & (dados_incompletos["local"] == "Interior")
]

n_concentrados = min(14, len(candidatos_concentrados))
faltantes_concentrados = rng_missing.choice(
    candidatos_concentrados,
    size=n_concentrados,
    replace=False,
)

restantes = dados_incompletos.index.difference(faltantes_concentrados)
faltantes_dispersos = rng_missing.choice(restantes, size=4, replace=False)

indices_faltantes = np.concatenate([
    np.asarray(faltantes_concentrados),
    np.asarray(faltantes_dispersos),
])

dados_incompletos.loc[indices_faltantes, "pessoas"] = np.nan

print("Valores ausentes por coluna:")
display(dados_incompletos.isna().sum())

percentual = 100 * dados_incompletos["pessoas"].isna().mean()
print(f"{percentual:.1f}% dos valores de 'pessoas' estão ausentes.")


A porcentagem total informa **quanto** falta, mas não **onde** falta. Vamos localizar a ausência por período e local.


In [ ]:
dados_incompletos["periodo"] = np.where(
    dados_incompletos["ano"] <= 1899,
    "1890–1899",
    "1900–1909",
)

ausencia_por_grupo = pd.crosstab(
    [dados_incompletos["periodo"], dados_incompletos["local"]],
    dados_incompletos["pessoas"].isna(),
)

ausencia_por_grupo.columns = ["observado", "ausente"]
ausencia_por_grupo["proporcao_ausente"] = (
    ausencia_por_grupo["ausente"]
    / ausencia_por_grupo[["observado", "ausente"]].sum(axis=1)
)

ausencia_por_grupo.round(3)


Agora a interpretação é diferente: a ausência está **concentrada** em uma parte do corpus.

Dizer apenas “15% dos valores estão ausentes” ocultaria uma característica importante. Se a perda de dados estiver
associada ao período, local, gênero, suporte documental ou qualidade de preservação, análises baseadas apenas nos
casos completos podem representar alguns segmentos melhor do que outros.

Por isso, uma inspeção mínima de dados ausentes deve perguntar:

1. **quanto** está ausente?
2. **em quais variáveis**?
3. **em quais grupos ou períodos**?
4. a ausência pode estar relacionada ao fenômeno que queremos estudar?

O experimento acima existe apenas na memória do notebook. `dados/documentos_relacoes.csv` permanece inalterado.


## 8. Associação, explicação e causalidade

Os exemplos anteriores permitem distinguir três tipos de frase.

### Descrição

> “Neste corpus, documentos com mais páginas tendem a mencionar mais pessoas.”

Resume uma associação observada.

### Hipótese explicativa

> “A associação pode ocorrer porque documentos maiores oferecem mais oportunidades para mencionar pessoas.”

Propõe um mecanismo que pode ser investigado.

### Afirmação causal

> “Aumentar o número de páginas faz um documento mencionar mais pessoas.”

Afirma que uma mudança em uma variável **produz** mudança na outra. A correlação observacional, sozinha, não
demonstra isso.

A conhecida expressão **“correlação não implica causalidade”** é mais útil quando convertida em perguntas
concretas:

- existe uma terceira variável relevante?
- estamos comparando contagens ou taxas?
- a composição dos grupos muda?
- os dados ausentes estão concentrados?
- qual mecanismo justificaria uma relação causal?
- que evidência adicional permitiria distinguir explicações concorrentes?


## 9. Exercício — o pesquisador cético

Considere as afirmações abaixo.

**A.** “Ao longo dos anos, os textos ficaram maiores.”

**B.** “Documentos com mais páginas mencionam mais pessoas.”

**C.** “Um gênero documental menciona mais pessoas que outro.”

**D.** “Os documentos da Capital são diferentes dos documentos do Interior.”

Para **cada afirmação**, responda:

1. Que evidência do corpus poderia sustentá-la?
2. Há uma terceira variável que deveria ser considerada?
3. A comparação envolve uma **contagem** que talvez devesse ser normalizada?
4. A frase é **descritiva**, **explicativa** ou **causal**?
5. Que análise adicional você faria antes de aceitar uma interpretação mais forte?

Não é necessário encontrar uma única resposta correta. O objetivo é explicitar quais decisões transformam uma
estatística em uma interpretação.


## 10. Síntese

Nesta etapa, uma mesma ideia apareceu sob diferentes formas:

> **uma associação é sempre calculada dentro de uma determinada organização dos dados.**

Vimos que:

- uma contagem pode refletir a quantidade de **exposição**;
- normalizar por um denominador pode responder a uma pergunta diferente;
- uma terceira variável pode ajudar a explicar uma associação aparente;
- relações agregadas podem mudar de direção quando examinamos grupos relevantes;
- valores ausentes precisam ser localizados, não apenas contados;
- descrição, hipótese explicativa e afirmação causal não são equivalentes.

A estatística descritiva continua correta em todos esses casos. O desafio é saber **qual pergunta cada cálculo
realmente responde**.
